# 🧠 Morse Code Classifier — Developing a Tiny Neural Network

In this workshop, we’ll train a simple neural network to **decode Morse code signals** from "timing features", and later deploy it on an **STM32 microcontroller**.

This notebook walks through:
1. Importing libraries
2. Preprocessing the Morse dataset
3. Building a neural network in TensorFlow
4. Training and visualizing results
5. Making predictions for new Morse signals
6. Converting the model parameters into a C header file for the STM32


In [ ]:
# 🧩 Import required libraries
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.callbacks import EarlyStopping

## Step 1: Define the Morse Code Classifier Class

To keep our code clean and modular, we’ll wrap everything inside a class.

This helps us easily understand, reuse and modify the model if needed.


In [ ]:
class MorseCodeClassifier:
  def __init__(self):
      """Initialize model, scaler, and label encoder."""
      self.model = None
      self.scaler = StandardScaler()
      self.label_encoder = LabelEncoder()


# Step 2: Data Preprocessing

The dataset (`morse_code_data.csv`) contains timing measurements (in microseconds) and corresponding Morse characters.

Each row looks like this:
```
timing1, timing2, signal3, signal4, letter
447872,	138585,	549335,	133193, C
```
We'll:
1. Read the CSV file
2. Separate the timing features (X) and labels (y)
3. Encode the letters into numbers for training
4. Scale the timing data so the model trains faster


In [ ]:
def preprocess_data(self, data_path):
    # Read the CSV file
    df = pd.read_csv(data_path, header=None)

    # Extract features (first 4 columns) and labels (last column)
    X = df.iloc[:, :4].values
    y = df.iloc[:, 4].str.strip()  # Remove any whitespace

    # Encode string labels (e.g., A, B, C) to numbers
    y_encoded = self.label_encoder.fit_transform(y)

    # Standardize feature scales
    X_scaled = self.scaler.fit_transform(X)

    return X_scaled, y_encoded

# Add this method to the class
MorseCodeClassifier.preprocess_data = preprocess_data


# Step 3: Build the Neural Network

We’ll build a **simple feedforward neural network** with:
- Two hidden layers (16 neurons each + ReLU activation)
- Softmax output layer for multi-class classification

The last output neuron is reserved for “unclassified” Morse patterns.


In [ ]:
def build_model(self, input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(input_shape,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(num_classes + 1, activation='softmax')  # +1 = unclassified
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    self.model = model
    return model

MorseCodeClassifier.build_model = build_model


# Step 4: Train the Model

- We use **early stopping** to prevent overfitting.
- We also add a small amount of random "unclassified" data to make the model robust.
- Training and validation accuracy/loss will be plotted afterward.


In [ ]:
def train(self, X, y, validation_split=0.2, epochs=50, batch_size=32, patience=5):
    # Create 10% "unclassified" random samples
    X_unclassified = np.random.randn(len(X) // 10, X.shape[1])
    y_unclassified = np.full(len(X_unclassified), len(np.unique(y)))

    # Combine classified + unclassified data
    X_combined = np.vstack([X, X_unclassified])
    y_combined = np.concatenate([y, y_unclassified])

    # Split into train and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_combined, y_combined, test_size=validation_split, random_state=42
    )

    # Stop training when validation loss stops improving
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True
    )

    # Train model
    self.history = self.model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=1
    )

    self.plot_training_history()
    return self.history

MorseCodeClassifier.train = train

# Step 5: Plot Training History

We want to visualize how the model performed over time.
- Accuracy plot shows how well the model learned.
- Loss plot shows how well it minimized prediction error.


In [ ]:

def plot_training_history(self):
    """Plot accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(self.history.history['accuracy'], label='Training Accuracy')
    ax1.plot(self.history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(self.history.history['loss'], label='Training Loss')
    ax2.plot(self.history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

MorseCodeClassifier.plot_training_history = plot_training_history


# Step 6: Make Predictions

Once trained, we can feed in new Morse timing data and get the predicted letter.

The model scales the input, makes a prediction, and converts the output index back to a letter.


In [ ]:
def predict(self, timing_data):
    """Predict Morse code characters from timing data."""
    if len(timing_data.shape) == 1:
        timing_data = timing_data.reshape(1, -1)

    timing_data_scaled = self.scaler.transform(timing_data)
    predictions = self.model.predict(timing_data_scaled)

    predicted_classes = np.argmax(predictions, axis=1)

    results = []
    for pred_class in predicted_classes:
        if pred_class == len(self.label_encoder.classes_):
            results.append('U')  # Unclassified
        else:
            results.append(self.label_encoder.inverse_transform([pred_class])[0])

    return results, predictions

MorseCodeClassifier.predict = predict


# Step 7: Run Everything!

Let’s bring it all together:
1. Initialize the classifier
2. Preprocess the dataset
3. Build the model
4. Train it and visualize performance


In [ ]:
# Initialize classifier
classifier = MorseCodeClassifier()

# Load and preprocess data
X, y = classifier.preprocess_data('morse_code_data.csv')

# Build model
input_shape = X.shape[1]
num_classes = len(np.unique(y))
classifier.build_model(input_shape, num_classes)

# Train model
history = classifier.train(X, y, epochs=200, patience=5)


# Step 8: Test a Sample Morse Signal

Let's try predicting our own new signal:


In [ ]:
sample_input = np.array([0, 0, 0, 0])
predicted_letter, confidence = classifier.predict(sample_input)

print(f"Predicted Letter: {predicted_letter}")
print(f"Confidence: {np.max(confidence) * 100:.2f}%")

# Step 9: Extracting Model Parameters for the STM32

Now that training is complete, we’ll extract:

- **Weights and biases** from each hidden layer  
- **Input scaling parameters** (mean and scale) from the `StandardScaler`  

for direct use in our STM32 inference/prediction code.


In [ ]:
# Function to extract parameters from the trained model
def extract_network_parameters(model, scaler):

    params = {}

    # Input scaling
    params['input_mean'] = scaler.mean_.tolist()
    params['input_scale'] = scaler.scale_.tolist()

    # Extract Dense layer weights and biases
    layer_count = 1
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            weights, biases = layer.get_weights()
            params[f'weights{layer_count}'] = weights.tolist()
            params[f'biases{layer_count}'] = biases.tolist()
            layer_count += 1

    return params

# Extract parameters from our classifier
network_params = extract_network_parameters(classifier.model, classifier.scaler)

## Step 10: Generate C Header File

We'll now convert the extracted parameters into C code so they can be included in an STM32 project.


- Everything will be saved in `network_parameters.h`.

In [ ]:
# Function to convert extracted parameters to C arrays
def generate_c_parameters(params):

    c_code = []

    # Header comment
    c_code.append("// Auto-generated neural network parameters")
    c_code.append("// Generated from MorseCodeClassifier Notebook\n")

    # Input scaling
    c_code.append("// Input scaling parameters")
    c_code.append(f"const float input_mean[{len(params['input_mean'])}] = "
                  + "{" + ", ".join(f"{x:.6f}f" for x in params['input_mean']) + "};")
    c_code.append(f"const float input_scale[{len(params['input_scale'])}] = "
                  + "{" + ", ".join(f"{x:.6f}f" for x in params['input_scale']) + "};")

    # Dense layer parameters
    layer_index = 1
    while f'weights{layer_index}' in params:
        weights = params[f'weights{layer_index}']
        biases = params[f'biases{layer_index}']

        c_code.append(f"\n// ===== Layer {layer_index} =====")

        # Weights array
        c_code.append(f"const float weights{layer_index}[] = {{")
        for row in weights:
            c_code.append("    " + ", ".join(f"{x:.6f}f" for x in row) + ",")
        c_code.append("};")

        # Biases array
        c_code.append(f"const float biases{layer_index}[] = "
                      + "{" + ", ".join(f"{x:.6f}f" for x in biases) + "};")

        layer_index += 1

    return "\n".join(c_code)


# Generate C code
c_parameters = generate_c_parameters(network_params)

# Save to file
with open('network_parameters.h', 'w') as f:
    f.write("#ifndef NETWORK_PARAMETERS_H\n")
    f.write("#define NETWORK_PARAMETERS_H\n\n")
    f.write(c_parameters)
    f.write("\n\n#endif // NETWORK_PARAMETERS_H")

print("💾 'network_parameters.h' successfully generated.")
